In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import json
from scipy import stats
from nba_api.stats.endpoints import leaguedashteamstats
from datetime import datetime

pd.set_option('display.max_columns', None)

project_root = Path.cwd().parent.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.models.xgboost_model import *
from src.models.ngboost_model import *
from src.points_model import PointsPropModel
from src.utils.helper_functions import findOpp
from src.utils.team_info import projectedStartingFive, mainStartingFive, teamStarPlayer, nameDict, questionablePlayers

/Users/alexgonzalez/Documents/NBA-Prop-Predictor/nba_model/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
{'BKN': ['E.J. Liddell'], 'GSW': ['Seth Curry']}

Out Players:
{'WAS': ['Alex Sarr', 'Corey Kispert', 'Bilal Coulibaly', 'Khris Middleton', 'Malaki Branham'], 'IND': ['Obi Toppin', 'Ben Sheppard', 'Aaron Nesmith'], 'CHA': ['Grant Williams', 'Pat Connaughton', 'Collin Sexton', 'LaMelo Ball', 'Tre Mann'], 'CLE': ['Larry Nance', 'Sam Merrill', 'Evan Mobley', 'Max Strus'], 'PHI': ['Trendon Watford', 'Kelly Oubre', 'Tyrese Maxey'], 'ATL': ['Trae Young', 'Kristaps Porziņģis', "N'Faly Dante"], 'MIL': ['Taurean Prince', 'AJ Green', 'Giannis Antetokounmpo'], 'BKN': ['Ben Saraf', 'Cam Thomas', 'Haywood Highsmith'], 'NOP': ['Dejounte Murray'], 'CHI': ['Ayo Dosunmu'], 'SAC': ['Domantas Sabonis', 'Drew Eubanks'], 'MIN': ['Mike Conley', 'Anthony Edwards'], 'LAL': ['Austin Reaves', 'Maxi Kleber'], 'PHX': ['Jalen Green', 'Isaiah Livers'], 'GSW': ['Al Horford', 'Gary Payton'], 'POR': ['Jrue Holiday', 'Matisse Thybulle', 'Scoot Henderson', 'Blake Wesley']}
Note: PHI (76ers) has 4 

In [3]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

if us_file is None:
    raise FileNotFoundError(f"No NBA_US file found for {today}")
if dfs_file is None:
    raise FileNotFoundError(f"No NBA_DFS file found for {today}")

us_df = pd.read_csv(us_file)
us_points = us_df[us_df['CATEGORY'] == 'player_points'].copy()

dfs_df = pd.read_csv(dfs_file)
prizepicks_lines = dfs_df[dfs_df['BOOKMAKER'] == 'PrizePicks'].copy()
pp_points = prizepicks_lines[prizepicks_lines['CATEGORY'] == 'player_points'].copy()
print(f"Found {len(pp_points)//2} PrizePicks player point props")

print(f"DFS earliest pull: {dfs_df['DATA_PULLED_AT'].min()}")
print(f"DFS latest pull: {dfs_df['DATA_PULLED_AT'].max()}")
print(f"US earliest pull: {us_df['DATA_PULLED_AT'].min()}")
print(f"US latest pull: {us_df['DATA_PULLED_AT'].max()}")
us_df.head()

Found 76 PrizePicks player point props
DFS earliest pull: 2025-12-14 15:51:15
DFS latest pull: 2025-12-14 15:51:15
US earliest pull: 2025-12-14 15:50:05
US latest pull: 2025-12-14 15:50:05


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,FanDuel,player_points,Dyson Daniels,Over,15.5,-110,2025-12-14,2025-12-14T23:50:02Z,2025-12-14 15:50:05
1,FanDuel,player_points,Dyson Daniels,Under,15.5,-120,2025-12-14,2025-12-14T23:50:02Z,2025-12-14 15:50:05
2,FanDuel,player_points,Jared McCain,Over,4.5,-108,2025-12-14,2025-12-14T23:50:02Z,2025-12-14 15:50:05
3,FanDuel,player_points,Jared McCain,Under,4.5,-122,2025-12-14,2025-12-14T23:50:02Z,2025-12-14 15:50:05
4,FanDuel,player_points,Joel Embiid,Over,24.5,-108,2025-12-14,2025-12-14T23:50:02Z,2025-12-14 15:50:05


In [4]:
# =============================================================================
# 2. GET PRIZEPICKS LINES AND FIND BEST MATCHING US ODDS
# =============================================================================

def american_to_implied(odds):
    """Convert American odds to implied probability."""
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    return 100 / (odds + 100)

def get_best_us_odds(player_name, line, side, us_points_df):
    """
    Find the best US sportsbook odds that match the PrizePicks line.
    Returns (best_odds, best_book) or (-137, None) if no match found.
    """
    player_lines = us_points_df[
        (us_points_df['NAME'] == player_name) &
        (us_points_df['LINE'] == line) &
        (us_points_df['OVER/UNDER'] == side)
    ]
    
    if player_lines.empty:
        return -137, None  # Default to -137 if no match
    
    # Find best odds (highest = least negative or most positive)
    best_idx = player_lines['ODDS'].idxmax()
    best_odds = int(player_lines.loc[best_idx, 'ODDS'])
    best_book = player_lines.loc[best_idx, 'BOOKMAKER']
    
    return best_odds, best_book

def get_market_fair_prob(player_name, line, side, us_points_df):
    """
    Get fair probability from US sportsbook consensus.
    Averages implied probabilities across books and deducts vig.
    """
    player_lines = us_points_df[
        (us_points_df['NAME'] == player_name) &
        (us_points_df['LINE'] == line) &
        (us_points_df['OVER/UNDER'] == side)
    ]
    
    if player_lines.empty:
        return None
    
    # Average implied probability across all books
    implied_probs = player_lines['ODDS'].apply(american_to_implied)
    avg_implied = implied_probs.mean()
    
    # Deduct half the vig (~2.5% for -110/-110)
    fair_prob = avg_implied - 0.025
    
    return max(0.01, min(0.99, fair_prob))

# Build PrizePicks props with best matching US odds
pp_props = []

for player in pp_points['NAME'].unique():
    player_data = pp_points[pp_points['NAME'] == player]
    
    over_line = pp_points[(pp_points['NAME'] == player) & (pp_points['OVER/UNDER'] == 'Over')]
    under_line = pp_points[(pp_points['NAME'] == player) & (pp_points['OVER/UNDER'] == 'Under')]
    
    if over_line.empty:
        continue
        
    line = over_line.iloc[0]['LINE']
    pp_odds = over_line.iloc[0]['ODDS']  # PrizePicks odds (for reference)
    
    # Find best matching US odds for each side
    best_us_odds_over, best_us_book_over = get_best_us_odds(player, line, 'Over', us_points)
    best_us_odds_under, best_us_book_under = get_best_us_odds(player, line, 'Under', us_points)
    
    # Get market fair probabilities from US books
    fair_over = get_market_fair_prob(player, line, 'Over', us_points)
    fair_under = get_market_fair_prob(player, line, 'Under', us_points)
    
    pp_props.append({
        'player': player,
        'line': line,
        'pp_odds': pp_odds,
        'best_us_odds_over': best_us_odds_over,
        'best_us_book_over': best_us_book_over,
        'best_us_odds_under': best_us_odds_under,
        'best_us_book_under': best_us_book_under,
        'fair_prob_over': fair_over,
        'fair_prob_under': fair_under,
    })

pp_props_df = pd.DataFrame(pp_props)
print(f"\nBuilt {len(pp_props_df)} PrizePicks props with best matching US odds")


Built 76 PrizePicks props with best matching US odds


In [5]:
# =============================================================================
# 3. FIT POINTS MODEL AND GET PROJECTIONS
# =============================================================================

# Load game logs
s26 = pd.read_csv('data/processed/training/PTS_TRAIN_26.csv')

def parse_minutes(min_str):
    if pd.isna(min_str): return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

s26_prepped = s26.copy()
s26_prepped['minutes'] = s26_prepped['MIN'].apply(parse_minutes)
s26_prepped = s26_prepped.rename(columns={
    'PLAYER_ID': 'player_id', 'PLAYER_NAME': 'player_name',
    'FGA': 'fga', 'FG3A': 'fg3a', 'FTA': 'fta',
    'FGM': 'fgm', 'FG3M': 'fg3m', 'FTM': 'ftm',
    'PTS': 'pts', 'PLUS_MINUS': 'margin'
})

name_to_id = s26_prepped.groupby('player_name')['player_id'].first().to_dict()
name_to_team = s26_prepped.groupby('player_name')['TEAM_ABBREVIATION'].last().to_dict()

# Get team stats
league_df = leaguedashteamstats.LeagueDashTeamStats(
    league_id_nullable='00',
    per_mode_detailed='PerGame',
    measure_type_detailed_defense='Advanced'
).get_data_frames()[0]
team_stats = league_df.set_index('TEAM_ID')

In [6]:
# =============================================================================
# SMART USAGE ADJUSTMENT BASED ON STARTERS + POSITION + USAGE DATA
# Handles: OUT players, QUESTIONABLE players, and lineup changes
# =============================================================================
# 
# IMPORTANT: PTS_DELTA_STAR_OUT is a RAW POINTS DIFFERENCE, not a multiplier!
# Example: If player averages 15 pts with star, 18 pts without → delta = +3.0
# We need to convert this to a multiplier: 1 + (3.0 / 15.0) = 1.20
# =============================================================================

def normalize_name(name):
    """Normalize player name using nameDict for special characters."""
    return nameDict.get(name, name)

def parse_minutes(min_str):
    """Parse minutes from string format (MM:SS) or numeric."""
    if pd.isna(min_str): 
        return 0.0
    if ':' in str(min_str):
        parts = str(min_str).split(':')
        return int(parts[0]) + int(parts[1])/60
    return float(min_str)

# Build player stats cache from training data
def build_player_stats_cache(df):
    """
    Build a cache of player stats for usage adjustment calculations.
    
    Key insight: PTS_DELTA_STAR_OUT is raw points difference, not a multiplier.
    We need the player's baseline to convert it properly.
    """
    player_stats = {}
    
    for player_name in df['PLAYER_NAME'].unique():
        player_df = df[df['PLAYER_NAME'] == player_name]
        if len(player_df) < 5:
            continue
            
        # Get most recent data (last 15 games for stability)
        recent = player_df.tail(15)
        
        # Position (from binary columns)
        guard = recent['GUARD'].mean() > 0.5 if 'GUARD' in recent.columns else False
        forward = recent['FORWARD'].mean() > 0.5 if 'FORWARD' in recent.columns else False
        center = recent['CENTER'].mean() > 0.5 if 'CENTER' in recent.columns else False
        
        # Get usage rate
        usg_pct = recent['USG_PCT'].mean() if 'USG_PCT' in recent.columns else 0.20
        
        # Parse minutes properly
        if 'MIN' in recent.columns:
            mins_vals = recent['MIN'].apply(lambda x: parse_minutes(x) if pd.notna(x) else 0)
            minutes = mins_vals.mean()
        else:
            minutes = 20.0
        
        # Get average points (baseline for calculating boost multiplier)
        pts_avg = recent['PTS'].mean() if 'PTS' in recent.columns else 10.0
        
        # =================================================================
        # FIXED: Convert PTS_DELTA_STAR_OUT to a proper multiplier
        # =================================================================
        pts_boost_multiplier = 1.0  # Default: no boost
        
        if 'PTS_DELTA_STAR_OUT' in recent.columns:
            delta = recent['PTS_DELTA_STAR_OUT'].mean()
            
            if pd.notna(delta) and pts_avg > 0:
                # Convert delta to multiplier
                raw_multiplier = 1 + (delta / pts_avg)
                
                # Sanity check: cap between 0.70 and 1.40
                if 0.70 <= raw_multiplier <= 1.40:
                    pts_boost_multiplier = raw_multiplier
        
        # Also check for games without star sample size
        games_without_star = 0
        if 'GAMES_WITHOUT_STAR' in recent.columns:
            games_without_star = recent['GAMES_WITHOUT_STAR'].iloc[-1] if len(recent) > 0 else 0
        
        player_stats[player_name] = {
            'guard': guard,
            'forward': forward,
            'center': center,
            'usg_pct': usg_pct,
            'minutes': minutes,
            'pts_avg': pts_avg,
            'pts_boost_star_out': pts_boost_multiplier,
            'games_without_star': games_without_star,
            'team': recent['TEAM_ABBREVIATION'].iloc[-1] if 'TEAM_ABBREVIATION' in recent.columns else 'UNK',
            'sample_size': len(recent)
        }
    
    return player_stats

# Build the stats cache
player_stats_cache = build_player_stats_cache(s26)
print(f"Built player stats cache for {len(player_stats_cache)} players")

# Quick sanity check on boost values
boost_values = [v['pts_boost_star_out'] for v in player_stats_cache.values() if v['pts_boost_star_out'] != 1.0]
if boost_values:
    print(f"Players with star-out boost data: {len(boost_values)}")
    print(f"  Boost range: {min(boost_values):.2f}x to {max(boost_values):.2f}x")
    print(f"  Mean boost: {np.mean(boost_values):.2f}x")

# Helper functions
def get_player_position(player_name):
    """Get player position from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    positions = []
    if stats.get('guard', False):
        positions.append('guard')
    if stats.get('forward', False):
        positions.append('forward')
    if stats.get('center', False):
        positions.append('center')
    return positions if positions else ['unknown']

def is_same_position(player1, player2):
    """Check if two players share at least one position."""
    pos1 = set(get_player_position(player1))
    pos2 = set(get_player_position(player2))
    return bool(pos1 & pos2)

def is_ball_handler(player_name):
    """Check if player is a primary ball handler (guard with decent usage)."""
    positions = get_player_position(player_name)
    usg = get_player_usage_rate(player_name)
    return 'guard' in positions and usg > 0.18

def get_player_usage_rate(player_name):
    """Get player's usage rate from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('usg_pct', 0.20)

def get_player_minutes(player_name):
    """Get player's average minutes from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('minutes', 20.0)

def get_player_pts_avg(player_name):
    """Get player's average points from stats cache."""
    stats = player_stats_cache.get(normalize_name(player_name), {})
    return stats.get('pts_avg', 10.0)

def get_historical_boost(player_name):
    """
    Get historical boost multiplier when star is out.
    Already converted to multiplier format (e.g., 1.15 for 15% boost).
    """
    stats = player_stats_cache.get(normalize_name(player_name), {})
    boost = stats.get('pts_boost_star_out', 1.0)
    games = stats.get('games_without_star', 0)
    return boost, games

def get_game_spread(team_abbrev, current_date, team_lines_dir='data/raw/team_lines'):
    """
    Get the spread for a team's game on a given date.
    """
    # Convert date to file format (YYYYMMDD)
    date_str = current_date.replace('-', '')
    
    # Find all files matching the date pattern
    team_lines_path = Path(team_lines_dir)
    pattern = f'NBA_{date_str}_*.json'
    matching_files = list(team_lines_path.glob(pattern))
    
    if not matching_files:
        return None
    
    # Get the latest file (by modification time, or by time in filename)
    # Option 1: By modification time (most recent scrape)
    latest_file = max(matching_files, key=lambda p: p.stat().st_mtime)
    
    # Option 2: By time in filename (if you prefer)
    # latest_file = max(matching_files, key=lambda p: int(p.stem.split('_')[-1]) if p.stem.split('_')[-1].isdigit() else 0)
    
    try:
        with open(latest_file, 'r') as f:
            games_data = json.load(f)
        
        # Map team abbreviations to full names
        team_name_map = {
            'ATL': 'Atlanta Hawks', 'BOS': 'Boston Celtics', 'BKN': 'Brooklyn Nets',
            'CHA': 'Charlotte Hornets', 'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
            'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets', 'DET': 'Detroit Pistons',
            'GSW': 'Golden State Warriors', 'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
            'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers', 'MEM': 'Memphis Grizzlies',
            'MIA': 'Miami Heat', 'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
            'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks', 'OKC': 'Oklahoma City Thunder',
            'ORL': 'Orlando Magic', 'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
            'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings', 'SAS': 'San Antonio Spurs',
            'TOR': 'Toronto Raptors', 'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards'
        }
        
        team_full_name = team_name_map.get(team_abbrev)
        if not team_full_name:
            return None
        
        # Find the game with this team
        for game in games_data:
            is_home = game['home_team'] == team_full_name
            is_away = game['away_team'] == team_full_name
            
            if is_home or is_away:
                # Get spread from first bookmaker (or average across bookmakers)
                for bookmaker in game['bookmakers']:
                    for market in bookmaker['markets']:
                        if market['market_key'] == 'spreads':
                            for outcome in market['outcomes']:
                                if outcome['name'] == team_full_name:
                                    spread = outcome['point']
                                    # Spread is from team's perspective
                                    # Negative = favored (expected to win by that amount)
                                    # Positive = underdog (expected to lose by that amount)
                                    return spread
        return None
    except Exception as e:
        return None

def calculate_blowout_prob_from_spread(spread):
    """
    Calculate blowout probability from spread.
    
    Blowout = |actual margin| > 20
    Using spread as expected margin, calculate probability of blowout.
    """
    if spread is None:
        return 0.15  # Default fallback
    
    # Expected margin from team's perspective
    # If spread is -4.5, team is expected to win by 4.5
    # If spread is +4.5, team is expected to lose by 4.5
    expected_margin = -spread  # Flip sign: negative spread = positive margin
    
    # Typical NBA game margin std dev is ~12 points
    margin_std = 12.0
    
    # Probability of blowout = P(|margin| > 20)
    # This is P(margin > 20) + P(margin < -20)
    prob_win_blowout = 1 - stats.norm.cdf(20, expected_margin, margin_std)
    prob_loss_blowout = stats.norm.cdf(-20, expected_margin, margin_std)
    blowout_prob = prob_win_blowout + prob_loss_blowout
    
    # Clamp between reasonable bounds
    return max(0.05, min(0.40, blowout_prob))

def get_out_and_questionable_players(team_abbrev):
    """
    Get lists of out and questionable players for a team.
    Assumes outPlayers and questionablePlayers dicts are available globally.
    
    Returns:
        tuple: (list of out players, list of questionable players)
    """
    out = outPlayers.get(team_abbrev, [])
    questionable = questionablePlayers.get(team_abbrev, [])
    return out, questionable

def get_usage_adjustment(player_name, team_abbrev):
    """
    Smart usage adjustment based on:
    1. OUT players (100% certainty they won't play)
    2. QUESTIONABLE players (50% probability weighted adjustment)
    3. Projected vs main lineup changes
    4. Historical performance when stars out
    5. Position overlap for usage redistribution
    6. Starter status changes
    
    Returns:
        tuple: (adjustment_multiplier, reason_string)
    """
    adjustment = 1.0
    reasons = []
    
    # Normalize player name
    player_name_normalized = normalize_name(player_name)
    
    # Get lineups
    main_lineup = mainStartingFive.get(team_abbrev, [])
    projected = projectedStartingFive.get(team_abbrev, [])
    team_star = teamStarPlayer.get(team_abbrev)
    
    # Get out and questionable players
    out_players_raw, questionable_players_raw = get_out_and_questionable_players(team_abbrev)
    
    # Normalize names for comparison
    main_normalized = [normalize_name(p) for p in main_lineup]
    projected_normalized = [normalize_name(p) for p in projected]
    out_normalized = [normalize_name(p) for p in out_players_raw]
    questionable_normalized = [normalize_name(p) for p in questionable_players_raw]
    
    # =========================================================================
    # CHECK IF PLAYER IS OUT OR QUESTIONABLE
    # =========================================================================
    if player_name_normalized in out_normalized:
        return 0.0, "Player is OUT - do not bet"
    
    if player_name_normalized in questionable_normalized:
        # Questionable players get reduced projection
        adjustment *= 0.75
        reasons.append("Player is QUESTIONABLE: -25%")
    
    # =========================================================================
    # Find who's definitely out (confirmed + not in projected lineup)
    # =========================================================================
    confirmed_out = []
    
    # Players on injury report as OUT
    for player in out_players_raw:
        if normalize_name(player) not in projected_normalized:
            confirmed_out.append(player)
    
    # Players in main lineup but not in projected (and not questionable)
    for player in main_lineup:
        player_norm = normalize_name(player)
        if (player_norm not in projected_normalized and 
            player_norm not in out_normalized and
            player_norm not in questionable_normalized):
            confirmed_out.append(player)
    
    # Check player's starting status
    player_in_main = player_name_normalized in main_normalized
    player_in_projected = player_name_normalized in projected_normalized
    
    # =========================================================================
    # CASE 1: No one is out - only check starter status
    # =========================================================================
    if not confirmed_out and not questionable_players_raw:
        if player_in_main and not player_in_projected:
            # Player usually starts but NOT starting tonight
            adjustment *= 0.85
            reasons.append("Not starting tonight: -15%")
        elif player_in_projected and not player_in_main:
            # Player starting tonight but usually doesn't
            adjustment *= 1.08
            reasons.append("Starting tonight (expanded): +8%")
        
        adjustment = min(1.30, max(0.70, adjustment))
        return adjustment, "; ".join(reasons) if reasons else "Full strength"
    
    # =========================================================================
    # CASE 2: Someone is out or questionable - calculate boost
    # =========================================================================
    
    # Check if star player is affected
    star_is_out = team_star and normalize_name(team_star) in [normalize_name(p) for p in confirmed_out]
    star_is_questionable = team_star and normalize_name(team_star) in questionable_normalized
    
    # Check for historical data (only use if star is definitely out)
    historical_boost, games_sample = get_historical_boost(player_name)
    has_reliable_historical = (
        games_sample >= 3 and  # At least 3 games without star
        0.85 <= historical_boost <= 1.35 and  # Reasonable range
        historical_boost != 1.0  # Actually has data
    )
    
    if has_reliable_historical and star_is_out:
        # Use historical data with regression toward 1.0
        sample_weight = min(0.8, games_sample / 10)  # Max 80% weight on historical
        regressed_boost = historical_boost * sample_weight + 1.0 * (1 - sample_weight)
        
        adjustment *= regressed_boost
        boost_pct = (historical_boost - 1) * 100
        reasons.append(f"Star out history ({games_sample}g): {boost_pct:+.0f}% → {(regressed_boost-1)*100:+.0f}% regressed")
    
    elif has_reliable_historical and star_is_questionable:
        # Star is questionable: apply 50% of the historical boost
        sample_weight = min(0.8, games_sample / 10)
        regressed_boost = historical_boost * sample_weight + 1.0 * (1 - sample_weight)
        partial_boost = 1.0 + (regressed_boost - 1.0) * 0.5  # 50% of boost
        
        adjustment *= partial_boost
        boost_pct = (historical_boost - 1) * 100
        reasons.append(f"Star questionable ({games_sample}g history): {boost_pct:+.0f}% → {(partial_boost-1)*100:+.0f}% (50% weighted)")
    
    else:
        # No reliable historical data - estimate from usage redistribution
        
        # Process CONFIRMED OUT players (100% weight)
        for out_player in confirmed_out:
            if normalize_name(out_player) == player_name_normalized:
                continue
            out_usage = get_player_usage_rate(out_player)

            # Determine capture rate based on relationship
            if is_same_position(player_name, out_player):
                capture_rate = 0.30  # Same position captures most
                reason_tag = "same pos"
            elif is_ball_handler(player_name) and is_ball_handler(out_player):
                capture_rate = 0.25
                reason_tag = "ball handler"
            elif player_in_projected:
                capture_rate = 0.12  # Starters get indirect boost
                reason_tag = "starter"
            else:
                capture_rate = 0.05  # Bench minimal
                reason_tag = "bench"
            
            # Convert to boost multiplier
            player_usage = get_player_usage_rate(player_name)
            if player_usage > 0.05:
                usage_gained = out_usage * capture_rate
                boost_pct = usage_gained / player_usage
                boost_multiplier = 1 + min(0.18, boost_pct)  # Cap at 18% from any single player
                
                if boost_multiplier > 1.02:
                    adjustment *= boost_multiplier
                    reasons.append(f"{out_player} OUT ({reason_tag}): +{(boost_multiplier-1)*100:.0f}%")
        
        # Process QUESTIONABLE players (50% weight - they might play)
        for q_player in questionable_players_raw:
            q_player_norm = normalize_name(q_player)
            
            # Skip if player already in confirmed out list
            # Skip if this is the player themselves (they can't get a boost from being questionable)
            if q_player_norm == player_name_normalized:
                continue
            
            # Skip if player already in confirmed out list
            if q_player_norm in [normalize_name(p) for p in confirmed_out]:
                continue

            q_usage = get_player_usage_rate(q_player)
            
            # Determine capture rate (same logic as above)
            if is_same_position(player_name, q_player):
                capture_rate = 0.30
                reason_tag = "same pos"
            elif is_ball_handler(player_name) and is_ball_handler(q_player):
                capture_rate = 0.25
                reason_tag = "ball handler"
            elif player_in_projected:
                capture_rate = 0.12
                reason_tag = "starter"
            else:
                capture_rate = 0.05
                reason_tag = "bench"
            
            # Apply 50% probability weight for questionable status
            player_usage = get_player_usage_rate(player_name)
            if player_usage > 0.05:
                usage_gained = q_usage * capture_rate * 0.5  # 50% weight
                boost_pct = usage_gained / player_usage
                boost_multiplier = 1 + min(0.09, boost_pct)  # Cap at 9% (half of 18%)
                
                if boost_multiplier > 1.01:
                    adjustment *= boost_multiplier
                    reasons.append(f"{q_player} QUESTIONABLE ({reason_tag}): +{(boost_multiplier-1)*100:.0f}% (50% weighted)")
    
    # =========================================================================
    # Starter status adjustments (apply on top of above)
    # =========================================================================
    if player_in_main and not player_in_projected:
        adjustment *= 0.85
        reasons.append("Not starting tonight: -15%")
    elif player_in_projected and not player_in_main:
        adjustment *= 1.05
        reasons.append("Expanded role: +5%")
    
    # Final cap
    adjustment = min(1.30, max(0.0, adjustment))  # Allow 0.0 for OUT players
    
    return adjustment, "; ".join(reasons) if reasons else "No adjustment"

# =============================================================================
# Calculate usage adjustments for all players
# =============================================================================
usage_adjustments = {}
print("\n" + "="*80)
print("SMART USAGE ADJUSTMENTS (OUT + QUESTIONABLE PLAYERS)")
print("="*80)

# Show injury report summary
total_out = sum(len(players) for players in outPlayers.values())
total_questionable = sum(len(players) for players in questionablePlayers.values())
print(f"\n📋 Injury Report: {total_out} OUT, {total_questionable} QUESTIONABLE")

if total_out > 0:
    print("\n🚫 OUT Players:")
    for team, players in outPlayers.items():
        if players:
            print(f"  {team}: {', '.join(players)}")

if total_questionable > 0:
    print("\n❓ QUESTIONABLE Players:")
    for team, players in questionablePlayers.items():
        if players:
            print(f"  {team}: {', '.join(players)}")

print("\n" + "="*80)

adjusted_players = []
out_player_list = []

for player in pp_props_df['player'].unique():
    team = name_to_team.get(player, 'UNK')
    if team == 'UNK':
        usage_adjustments[player] = (1.0, "Unknown team")
        continue
        
    adj, reason = get_usage_adjustment(player, team)
    usage_adjustments[player] = (adj, reason)
    
    # Track players who are out
    if adj == 0.0:
        out_player_list.append({
            'player': player,
            'team': team,
            'reason': reason
        })
    elif adj != 1.0 and reason not in ["Full strength", "No adjustment"]:
        adjusted_players.append({
            'player': player,
            'team': team,
            'adjustment': adj,
            'reason': reason,
            'position': "/".join(get_player_position(player)),
            'usg_pct': get_player_usage_rate(player),
            'pts_avg': get_player_pts_avg(player)
        })

# Summary
boosts = [p for p in adjusted_players if p['adjustment'] > 1.0]
reductions = [p for p in adjusted_players if p['adjustment'] < 1.0]

print(f"\n📊 Summary: {len(out_player_list)} OUT, {len(boosts)} boosted, {len(reductions)} reduced, {len(pp_props_df['player'].unique()) - len(adjusted_players) - len(out_player_list)} unchanged")

if out_player_list:
    print(f"\n🚫 PLAYERS OUT (DO NOT BET) - {len(out_player_list)}:")
    for p in out_player_list:
        print(f"  ✗ {p['player']} ({p['team']}) - {p['reason']}")

if boosts:
    print(f"\n📈 BOOSTED ({len(boosts)}):")
    for p in sorted(boosts, key=lambda x: x['adjustment'], reverse=True)[:15]:
        print(f"  ↑ {p['player']} ({p['team']}) {p['adjustment']:.2f}x")
        print(f"     {p['position']}, USG:{p['usg_pct']:.1%}, Avg:{p['pts_avg']:.1f}pts")
        print(f"     {p['reason']}")

if reductions:
    print(f"\n📉 REDUCED ({len(reductions)}):")
    for p in sorted(reductions, key=lambda x: x['adjustment'])[:10]:
        print(f"  ↓ {p['player']} ({p['team']}) {p['adjustment']:.2f}x")
        print(f"     {p['reason']}")

Built player stats cache for 448 players
Players with star-out boost data: 194
  Boost range: 0.70x to 1.40x
  Mean boost: 1.03x

SMART USAGE ADJUSTMENTS (OUT + QUESTIONABLE PLAYERS)

📋 Injury Report: 45 OUT, 2 QUESTIONABLE

🚫 OUT Players:
  WAS: Alex Sarr, Corey Kispert, Bilal Coulibaly, Khris Middleton, Malaki Branham
  IND: Obi Toppin, Ben Sheppard, Aaron Nesmith
  CHA: Grant Williams, Pat Connaughton, Collin Sexton, LaMelo Ball, Tre Mann
  CLE: Larry Nance, Sam Merrill, Evan Mobley, Max Strus
  PHI: Trendon Watford, Kelly Oubre, Tyrese Maxey
  ATL: Trae Young, Kristaps Porziņģis, N'Faly Dante
  MIL: Taurean Prince, AJ Green, Giannis Antetokounmpo
  BKN: Ben Saraf, Cam Thomas, Haywood Highsmith
  NOP: Dejounte Murray
  CHI: Ayo Dosunmu
  SAC: Domantas Sabonis, Drew Eubanks
  MIN: Mike Conley, Anthony Edwards
  LAL: Austin Reaves, Maxi Kleber
  PHX: Jalen Green, Isaiah Livers
  GSW: Al Horford, Gary Payton
  POR: Jrue Holiday, Matisse Thybulle, Scoot Henderson, Blake Wesley

❓ QUESTI

In [7]:
# =============================================================================
# Fit model and get projections
# =============================================================================
model = PointsPropModel(min_edge=0.02, min_confidence=0.52)
model.fit(s26_prepped)

# Create mapping from team abbreviation to team_id
# Use s26_prepped which has both TEAM_ABBREVIATION and TEAM_ID
team_abbrev_to_id = s26_prepped.groupby('TEAM_ABBREVIATION')['TEAM_ID'].first().to_dict()

player_projections = {}

# Then in your projection loop (around line 997-1012):
for _, row in pp_props_df.iterrows():
    player = row['player']
    player_id = name_to_id.get(player)
    
    if player_id is None:
        continue
    
    usage_adj, adj_reason = usage_adjustments.get(player, (1.0, "No adjustment"))
    
    # Skip players who are OUT
    if usage_adj == 0.0:
        continue
    
    # Calculate if it's a back-to-back game
    player_games = s26_prepped[s26_prepped['player_id'] == player_id].sort_values('GAME_DATE')
    is_b2b = False
    if not player_games.empty:
        latest_game_date = pd.to_datetime(player_games['GAME_DATE'].iloc[-1])
        current_date_dt = pd.to_datetime(current_date)
        days_since_last_game = (current_date_dt - latest_game_date).days
        is_b2b = (days_since_last_game == 1)
    
    # Calculate blowout probability from spread
    player_team_abbrev = name_to_team.get(player)
    spread = get_game_spread(player_team_abbrev, current_date) if player_team_abbrev else None
    blowout_prob = calculate_blowout_prob_from_spread(spread)
    
    # Get opponent team abbreviation using findOpp
    opp_team_id = None
    try:
        opp_abbrev, _ = findOpp(player, s26, current_date)
        if opp_abbrev and opp_abbrev in team_abbrev_to_id:
            opp_team_id = int(team_abbrev_to_id[opp_abbrev])
    except Exception as e:
        pass
        
    try:
        projection = model.project_points(
            player_id=player_id,
            is_b2b=is_b2b,
            blowout_prob=blowout_prob, 
            usage_adjustment=usage_adj,
            opp_team_id=opp_team_id
        )
        if projection:
            player_projections[player] = {
                'expected': projection['expected_points'],
                'std': projection['std'],
                'team': name_to_team.get(player, 'UNK'),
                'usage_adj': usage_adj,
                'usage_reason': adj_reason
            }
    except Exception as e:
        continue

In [8]:
# =============================================================================
# 4. CALCULATE SINGLE LEG EDGES (USING BEST US ODDS FOR EV)
# =============================================================================

def calc_model_prob(projection, std, line, side):
    """Calculate model probability for over/under."""
    z = (line - projection) / std
    if side.lower() == 'over':
        return 1 - stats.norm.cdf(z)
    return stats.norm.cdf(z)

def calc_ev(model_prob, odds):
    """Calculate Expected Value from model probability and odds."""
    if odds < 0:
        decimal_odds = 100 / abs(odds) + 1
    else:
        decimal_odds = odds / 100 + 1
    ev = (model_prob * decimal_odds) - 1
    return ev

def calc_kelly(model_prob, odds, fraction=0.25):
    """Calculate Kelly Criterion (default quarter Kelly for safety)."""
    if odds < 0:
        decimal_odds = 100 / abs(odds) + 1
    else:
        decimal_odds = odds / 100 + 1
    
    kelly = (model_prob * decimal_odds - 1) / (decimal_odds - 1)
    kelly_fractional = max(0, kelly * fraction)
    return kelly_fractional

def american_to_implied(odds):
    """Convert American odds to implied probability."""
    if odds < 0:
        return abs(odds) / (abs(odds) + 100)
    else:
        return 100 / (odds + 100)

def calculate_hit_rate(player_name, line, side, data_df, windows=[5, 10, 15]):
    """
    Calculate hit rate percentage for a player hitting a line (over/under) in last N games.
    
    Args:
        player_name: Player name
        line: The line value
        side: 'Over' or 'Under'
        data_df: DataFrame with historical game data (should have PLAYER_NAME, PTS, GAME_DATE)
        windows: List of game windows to calculate (default: [5, 10, 15])
    
    Returns:
        Dictionary with hit rates for each window (as percentages)
    """
    # Filter to player's games
    player_df = data_df[data_df['PLAYER_NAME'] == player_name].copy()
    
    if len(player_df) == 0:
        return {f'L-{w}': None for w in windows}
    
    # Sort by date (most recent last)
    player_df = player_df.sort_values('GAME_DATE')
    
    results = {}
    
    for window in windows:
        # Get last N games
        if len(player_df) < window:
            last_n_games = player_df
            actual_window = len(player_df)
        else:
            last_n_games = player_df.tail(window)
            actual_window = window
        
        if actual_window == 0:
            results[f'L-{window}'] = None
            continue
        
        # Calculate hits based on side
        if side.lower() == 'over':
            hits = (last_n_games['PTS'] > line).sum()
        else:  # under
            hits = (last_n_games['PTS'] < line).sum()
        
        # Calculate percentage
        hit_rate_pct = (hits / actual_window) * 100
        results[f'L-{window}'] = round(hit_rate_pct, 1)
    
    return results

single_bets = []

for _, row in pp_props_df.iterrows():
    player = row['player']
    line = row['line']
    pp_odds = row['pp_odds']
    
    if player not in player_projections:
        continue
        
    proj = player_projections[player]
    
    # Model probabilities
    model_prob_over = calc_model_prob(proj['expected'], proj['std'], line, 'over')
    model_prob_under = calc_model_prob(proj['expected'], proj['std'], line, 'under')
    
    # Market fair probabilities (from US books)
    fair_over = row.get('fair_prob_over', 0.5) or 0.5
    fair_under = row.get('fair_prob_under', 0.5) or 0.5
    
    # PrizePicks implied probability
    pp_implied = american_to_implied(pp_odds)
    
    # Calculate edges for both sides
    edge_over = model_prob_over - fair_over
    edge_under = model_prob_under - fair_under
    
    # Edge vs PrizePicks line (50% breakeven for DFS)
    edge_vs_pp_over = model_prob_over - 0.50
    edge_vs_pp_under = model_prob_under - 0.50
    
    # Determine best side based on edge vs fair market
    if edge_over > edge_under:
        best_side = 'Over'
        best_model_prob = model_prob_over
        best_fair_prob = fair_over
        best_edge = edge_over
        best_edge_vs_pp = edge_vs_pp_over
        best_us_odds = row.get('best_us_odds_over', pp_odds)
        best_us_book = row.get('best_us_book_over', 'PrizePicks')
    else:
        best_side = 'Under'
        best_model_prob = model_prob_under
        best_fair_prob = fair_under
        best_edge = edge_under
        best_edge_vs_pp = edge_vs_pp_under
        best_us_odds = row.get('best_us_odds_under', pp_odds)
        best_us_book = row.get('best_us_book_under', 'PrizePicks')
    
    # Handle missing US odds - use PrizePicks default
    if pd.isna(best_us_odds) or best_us_odds is None:
        best_us_odds = pp_odds
        best_us_book = f'Default ({pp_odds})'
    
    # Calculate EV and Kelly using best available odds
    ev = calc_ev(best_model_prob, best_us_odds)
    kelly_quarter = calc_kelly(best_model_prob, best_us_odds, fraction=0.25)
    
    # Confidence flag based on projection vs line gap
    pts_diff = abs(proj['expected'] - line)
    confidence = 'HIGH' if pts_diff > 1.5 * proj['std'] else 'MEDIUM' if pts_diff > proj['std'] else 'LOW'
    
    # Calculate hit rates for the best side
    hit_rates = calculate_hit_rate(player, line, best_side, s26)
    
    single_bets.append({
        'player': player,
        'team': proj['team'],
        'line': line,
        'side': best_side,
        'projection': round(proj['expected'], 1),
        'std': round(proj['std'], 1),
        'model_prob': round(best_model_prob, 3),
        'fair_prob': round(best_fair_prob, 3) if best_fair_prob else None,
        'pp_odds': pp_odds,
        'pp_implied': round(pp_implied, 3),
        'best_us_odds': int(best_us_odds) if not pd.isna(best_us_odds) else pp_odds,
        'best_us_book': best_us_book if best_us_book else f'Default ({pp_odds})',
        'edge_vs_fair': round(best_edge, 4) if best_fair_prob else None,
        'edge_vs_pp': round(best_edge_vs_pp, 4),
        'ev': round(ev, 4),
        'ev_percent': round(ev * 100, 2),
        'kelly_quarter': round(kelly_quarter, 2),
        'confidence': confidence,
        'edge_over': round(edge_over, 4),
        'edge_under': round(edge_under, 4),
        'usage_adj': proj.get('usage_adj', 1.0),
        'usage_reason': proj.get('usage_reason', 'No adjustment'),
        'L-5': hit_rates.get('L-5'),
        'L-10': hit_rates.get('L-10'),
        'L-15': hit_rates.get('L-15'),
    })

singles_df = pd.DataFrame(single_bets)

# Sort by edge and filter
singles_df = singles_df.sort_values('edge_vs_fair', ascending=False, na_position='last')

# Display top picks
print("\n" + "="*120)
print("TOP SINGLE PRIZEPICKS PICKS (sorted by edge vs market fair value)")
print("="*120)
print(f"{'Player':<25} {'Team':<6} {'Side':<6} {'Line':<6} {'Proj':<7} {'Model%':<8} {'Fair%':<8} {'Edge%':<8} {'US Odds':<18} {'EV%':<8} {'Kelly':<8} {'Conf':<6}")
print("-"*120)

for _, bet in singles_df.head(20).iterrows():
    fair_str = f"{bet['fair_prob']*100:.1f}%" if bet['fair_prob'] else "N/A"
    edge_str = f"{bet['edge_vs_fair']*100:.1f}%" if bet['edge_vs_fair'] else f"{bet['edge_vs_pp']*100:.1f}%*"
    us_odds_str = f"{bet['best_us_odds']:+d}" if bet['best_us_odds'] >= 0 else f"{bet['best_us_odds']}"
    
    print(f"{bet['player']:<25} {bet['team']:<6} {bet['side']:<6} {bet['line']:<6.1f} {bet['projection']:<7.1f} "
          f"{bet['model_prob']*100:<8.1f} {fair_str:<8} {edge_str:<8} {us_odds_str:<6} ({bet['best_us_book']:<10}) "
          f"{bet['ev_percent']:<8.1f} {bet['kelly_quarter']*100:<8.2f} {bet['confidence']:<6}")

# Show usage-adjusted players separately
adjusted_bets = singles_df[singles_df['usage_adj'] != 1.0]
if len(adjusted_bets) > 0:
    print("\n" + "="*80)
    print("USAGE-ADJUSTED PLAYERS IN TOP PICKS")
    print("="*80)
    for _, bet in adjusted_bets.head(10).iterrows():
        direction = "↑" if bet['usage_adj'] > 1 else "↓"
        print(f"{direction} {bet['player']} ({bet['team']}): {bet['usage_adj']:.2f}x")
        print(f"   Reason: {bet['usage_reason']}")
        print(f"   Pick: {bet['side']} {bet['line']} (Proj: {bet['projection']}, Edge: {bet['edge_vs_pp']*100:.1f}%)")


TOP SINGLE PRIZEPICKS PICKS (sorted by edge vs market fair value)
Player                    Team   Side   Line   Proj    Model%   Fair%    Edge%    US Odds            EV%      Kelly    Conf  
------------------------------------------------------------------------------------------------------------------------
Jeremiah Fears            NOP    Over   13.5   18.6    89.6     51.2%    38.5%    -110   (Bovada    ) 71.1     20.00    MEDIUM
Zion Williamson           NOP    Over   17.5   22.6    88.0     49.7%    38.3%    +105   (Bovada    ) 80.3     19.00    MEDIUM
Dillon Brooks             PHX    Over   18.5   24.3    88.6     50.8%    37.8%    -106   (DraftKings) 72.2     19.00    MEDIUM
Rui Hachimura             LAL    Over   12.5   16.3    84.7     48.5%    36.2%    +102   (FanDuel   ) 71.0     17.00    MEDIUM
Collin Gillespie          PHX    Over   12.5   17.2    86.8     51.3%    35.5%    -111   (FanDuel   ) 65.0     18.00    MEDIUM
Julian Champagnie         SAS    Over   7.5    10.3

In [9]:
# =============================================================================
# 6. SAVE RESULTS
# =============================================================================

# Prepare output DataFrame
output_df = singles_df[[
    'player', 'team', 'line', 'side', 'projection', 'std',
    'model_prob', 'fair_prob', 'pp_odds', 'pp_implied',
    'best_us_odds', 'best_us_book',
    'edge_vs_fair', 'edge_vs_pp', 'ev', 'ev_percent', 'kelly_quarter',
    'usage_adj', 'usage_reason', 'L-5', 'L-10', 'L-15'
]].rename(columns={
    'player': 'NAME',
    'team': 'TEAM',
    'line': 'LINE',
    'side': 'SIDE',
    'projection': 'PREDICTION',
    'std': 'STD',
    'model_prob': 'MODEL_PROB',
    'fair_prob': 'FAIR_PROB',
    'pp_odds': 'PRIZEPICKS_ODDS',
    'pp_implied': 'PRIZEPICKS_IMPLIED',
    'best_us_odds': 'BEST_US_ODDS',
    'best_us_book': 'BEST_US_BOOK',
    'edge_vs_fair': 'EDGE_VS_FAIR',
    'edge_vs_pp': 'EDGE_VS_PP',
    'ev': 'EV',
    'ev_percent': 'EV_PERCENT',
    'kelly_quarter': 'KELLY_QUARTER',
    'usage_adj': 'USAGE_ADJ',
    'usage_reason': 'USAGE_REASON',
    'L-5': 'L5',
    'L-10': 'L10',
    'L-15': 'L15'
})

# Save
from datetime import datetime
today = datetime.now().strftime('%Y-%m-%d')
output_path = f'data/props/ev_analysis/prizepicks.csv'
output_df.to_csv(output_path, index=False)
print(f"\n✓ Saved to {output_path}")
print(f"✓ Total picks: {len(output_df)}")
print(f"✓ Picks with >2% edge: {len(singles_df[singles_df['edge_vs_fair'] > 0.02])}")
print(f"✓ Picks with >5% edge: {len(singles_df[singles_df['edge_vs_fair'] > 0.05])}")
print(f"✓ Picks using default -137: {len(singles_df[singles_df['best_us_book'] == 'Default (-137)'])}")
print(f"✓ Picks with usage adjustments: {len(singles_df[singles_df['usage_adj'] != 1.0])}")


✓ Saved to data/props/ev_analysis/prizepicks.csv
✓ Total picks: 70
✓ Picks with >2% edge: 42
✓ Picks with >5% edge: 40
✓ Picks using default -137: 24
✓ Picks with usage adjustments: 64


In [10]:
singles_df[['player', 'side', 'line', 'projection', 'model_prob','ev_percent', 'kelly_quarter', 'L-5', 'L-10', 'L-15']].sort_values('ev_percent', ascending=False).head(20)

,player,side,line,projection,model_prob,ev_percent,kelly_quarter,L-5,L-10,L-15
3,Zion Williamson,Over,17.5,22.6,0.880,80.30,0.19,60.0,70.0,70.0
29,Dillon Brooks,Over,18.5,24.3,0.886,72.17,0.19,60.0,70.0,60.0
6,Jeremiah Fears,Over,13.5,18.6,0.896,71.09,0.20,80.0,70.0,80.0
32,Rui Hachimura,Over,12.5,16.3,0.847,71.04,0.17,40.0,60.0,66.7
39,Deni Avdija,Over,23.5,30.0,0.871,70.01,0.18,60.0,60.0,60.0
10,Kevin Huerter,Over,9.5,12.8,0.789,69.73,0.15,40.0,50.0,60.0
31,Deandre Ayton,Over,14.5,18.8,0.826,69.40,0.17,20.0,50.0,46.7
68,Julian Champagnie,Over,7.5,10.3,0.835,66.91,0.17,80.0,80.0,73.3
27,Devin Booker,Over,21.5,28.3,0.868,65.02,0.18,20.0,40.0,46.7
33,Collin Gillespie,Over,12.5,17.2,0.868,64.99,0.18,60.0,80.0,66.7
